# USGS Earthquake Agent Streaming Demo

This notebook demonstrates the dedicated `usgs_earthquake_agent` using GAS streaming mode. It walks through every public skill advertised by the agent:

- Retrieve and export earthquake events
- Map earthquake activity
- Summarize earthquake activity
- Analyze earthquake patterns
- Generate alert-ready summaries
- Generate earthquake reports

The agent does not require a USGS API key. If you provide an OpenAI or GIBD key, the service may use LLM-assisted planning for vague requests; otherwise it uses deterministic request parsing and deterministic USGS/geospatial tools.

## Install and Import

In [ ]:
%pip install -q gas-client requests pandas geopandas matplotlib python-dotenv

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import base64
import json
import os

import requests
from dotenv import load_dotenv
from IPython.display import HTML, Image, Markdown, display

from gas_client import GasClient

## User Settings

Use your local server during development or switch `server_url` to the deployed GAS server. Model credentials are optional for this agent.

In [ ]:
project_root = Path.cwd()
if project_root.name == "examples_for_using_gas_services":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

server_url = "http://127.0.0.1:4042"
# server_url = "https://www.geospatial-agentic-services.online"

openai_api_key = os.getenv("OPENAI_API_KEY", "").strip()
gibd_api_key = os.getenv("GIBD_API_KEY", "").strip()

credentials = {}
if openai_api_key:
    credentials["OPENAI_API_KEY"] = openai_api_key
elif gibd_api_key:
    credentials["GIBD_API_KEY"] = gibd_api_key

task_credentials = credentials or None
timeout_seconds = 1800

client = GasClient(server_url, artifact_delivery="URL")
earthquake_agent = client.agent("usgs_earthquake_agent")

## Helpers

`run_streaming_task(...)` prints progress events and returns the final task result. `display_task_artifacts(...)` downloads returned artifacts and renders common formats inline.

In [ ]:
artifact_cache_dir = project_root / "examples_for_using_gas_services" / "downloaded_gas_artifacts" / "usgs_earthquake_agent"
artifact_cache_dir.mkdir(parents=True, exist_ok=True)


def absolute_url(url):
    if not url:
        return None
    if url.startswith("/"):
        return urljoin(server_url.rstrip("/") + "/", url)
    return url


def artifact_display_name(artifact):
    return artifact.get("filename") or artifact.get("name") or Path(urlparse(artifact.get("url", "")).path).name or "gas_artifact"


def artifact_suffix(artifact):
    name = artifact_display_name(artifact)
    url = artifact.get("url") or ""
    return Path(name).suffix.lower() or Path(urlparse(url).path).suffix.lower()


def download_or_decode_artifact(artifact):
    name = artifact_display_name(artifact)
    suffix = artifact_suffix(artifact)
    if not Path(name).suffix and suffix:
        name = f"{name}{suffix}"
    path = artifact_cache_dir / Path(name).name

    if artifact.get("data"):
        path.write_bytes(base64.b64decode(artifact["data"]))
        return path

    url = absolute_url(artifact.get("url"))
    if not url:
        raise ValueError("Artifact has neither URL nor encoded data.")
    if not path.exists():
        response = requests.get(url, timeout=300)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path


def display_task_artifacts(task_result, *, max_rows=8, only_suffixes=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    if not artifacts:
        print("No artifacts returned.")
        return []

    displayed_paths = []
    for artifact in artifacts:
        name = artifact_display_name(artifact)
        suffix = artifact_suffix(artifact)
        if only_suffixes and suffix not in only_suffixes:
            continue

        print(f"Artifact: {name}")
        try:
            path = download_or_decode_artifact(artifact)
            displayed_paths.append(path)

            if suffix in {".geojson", ".gpkg", ".shp"}:
                import geopandas as gpd

                gdf = gpd.read_file(path)
                display(gdf.head(max_rows))
            elif suffix == ".csv":
                import pandas as pd

                display(pd.read_csv(path).head(max_rows))
            elif suffix == ".parquet":
                import pandas as pd

                display(pd.read_parquet(path).head(max_rows))
            elif suffix in {".png", ".jpg", ".jpeg", ".gif"}:
                display(Image(filename=str(path)))
            elif suffix in {".html", ".htm"}:
                html_text = path.read_text(encoding="utf-8", errors="ignore")
                display(HTML(f'<iframe srcdoc={html_text!r} width="100%" height="760" style="border:1px solid #ddd;"></iframe>'))
            elif suffix in {".md", ".markdown"}:
                display(Markdown(path.read_text(encoding="utf-8", errors="ignore")))
            elif suffix == ".json":
                display(json.loads(path.read_text(encoding="utf-8")))
            else:
                url = absolute_url(artifact.get("url"))
                if url:
                    display(HTML(f'<a href="{url}" target="_blank">Open or download {name}</a>'))
                else:
                    print(path)
        except Exception as exc:
            print(f"Could not render {name} inline: {exc}")
            url = absolute_url(artifact.get("url"))
            if url:
                display(HTML(f'<a href="{url}" target="_blank">Open or download {name}</a>'))
    return displayed_paths


def run_streaming_task(instructions, *, parameters=None, show_summary=True):
    final_result = None
    for event in earthquake_agent.execute_task(
        instructions,
        mode="stream",
        artifact_delivery="URL",
        parameters=parameters,
        credentials=task_credentials,
        timeout=timeout_seconds,
    ):
        client.print_stream_event(event)
        if event.get("event") == "task_result":
            final_result = event.get("payload")

    if final_result is None:
        raise RuntimeError("Streaming completed without a task_result event.")

    if show_summary:
        client.print_task_summary(final_result)
    return final_result


def show_earthquake_summary(task_result):
    summary = task_result.get("outputs", {}).get("data_summary", {})
    print("Standard data summary:")
    display(summary)
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    print(f"Artifact count: {len(artifacts)}")
    for artifact in artifacts:
        print("-", artifact.get("name"), artifact.get("format"), artifact.get("url"))

## 1. Retrieve And Export Earthquake Events

This demonstrates `retrieve_and_export_earthquake_events`: the agent queries USGS and returns a reusable earthquake event dataset artifact. Here we explicitly request CSV so the export behavior is visible.

In [ ]:
retrieve_export_result = run_streaming_task(
    "Retrieve M2.5+ earthquakes in California during the past 30 days and export the event dataset as CSV.",
    parameters={"output_format": "csv"},
)

show_earthquake_summary(retrieve_export_result)
display_task_artifacts(retrieve_export_result, only_suffixes={".csv", ".md"})

## 2. Map Earthquake Activity

This demonstrates `map_earthquake_activity`: the agent creates static PNG and interactive HTML basemap maps, plus an animation-ready GeoJSON artifact with timestamp/style fields.

In [ ]:
map_result = run_streaming_task(
    "Map M2.5+ earthquakes in Southern California during the past 30 days. Create a depth-colored map with a basemap and time-animation-ready GeoJSON.",
    parameters={"output_format": "geojson"},
)

display_task_artifacts(map_result, only_suffixes={".html", ".png", ".geojson", ".md"})

## 3. Summarize Earthquake Activity

This demonstrates `summarize_earthquake_activity`: the agent computes event counts, top events, magnitude/depth summaries, and a grid summary layer.

In [ ]:
summary_result = run_streaming_task(
    "Summarize M3+ earthquake activity in Alaska during the past 30 days. Include top events, magnitude and depth summaries, and a grid summary layer.",
    parameters={"output_format": "geojson", "grid_degrees": 2.0},
)

show_earthquake_summary(summary_result)
display_task_artifacts(summary_result, only_suffixes={".geojson", ".csv", ".md"})

## 4. Analyze Earthquake Patterns

This demonstrates `analyze_earthquake_patterns`: the agent creates impact buffers and performs simple cluster screening. These are screening artifacts for geospatial workflows, not formal seismic hazard products.

In [ ]:
pattern_result = run_streaming_task(
    "Analyze M2.5+ earthquake patterns in California during the past 30 days. Create impact buffers and screen for clusters.",
    parameters={"output_format": "geojson", "buffer_km": 50},
)

display_task_artifacts(pattern_result, only_suffixes={".geojson", ".png", ".md"})

## 5. Generate Alert-Ready Summary

This demonstrates `generate_alert_ready_summary`: the agent uses a real-time USGS feed or thresholded catalog query and produces concise monitoring text.

In [ ]:
alert_result = run_streaming_task(
    "Run a latest significant earthquake check and produce an alert-ready summary for the past day.",
    parameters={"feed_period": "day", "feed_magnitude": "significant", "output_format": "geojson"},
)

alert_summary = alert_result.get("outputs", {}).get("summary")
display(Markdown(f"### Alert-ready summary\n\n{alert_summary}"))
display_task_artifacts(alert_result, only_suffixes={".geojson", ".csv", ".md"})

## 6. Generate Earthquake Report

This demonstrates `generate_earthquake_report`: the agent returns Markdown and HTML reports with maps, charts, event tables, methods/provenance, and limitations.

In [ ]:
report_result = run_streaming_task(
    "Generate an HTML earthquake activity report for M4.5+ earthquakes near Japan during the past 30 days. Include maps, charts, top events, methods, provenance, and limitations.",
    parameters={"output_format": "geojson"},
)

display_task_artifacts(report_result, only_suffixes={".html", ".md", ".png", ".csv"})

## Artifact URLs

The returned URLs can be passed to other GAS agents, such as the Mapping Agent, Web Mapping App Agent, Spatial Statistics Agent, or Vector Analysis Agent.

In [ ]:
all_results = {
    "retrieve_export": retrieve_export_result,
    "map": map_result,
    "summary": summary_result,
    "patterns": pattern_result,
    "alert": alert_result,
    "report": report_result,
}

for label, result in all_results.items():
    print(f"\n{label}")
    for artifact in result.get("outputs", {}).get("artifacts", []):
        print("-", artifact.get("format"), artifact.get("name"), absolute_url(artifact.get("url")))